In [ ]:
import Pkg
Pkg.activate(".")
#Pkg.add(["FUSE", "IMAS", "Plots"])
Pkg.instantiate()
using FUSE, IMAS, Plots

In [ ]:
# Modify ARC V1 use case to match Sorbom 2015
ini,act = FUSE.case_parameters(:ARC)

# shaping
ini.equilibrium.B0 = -9.2
ini.equilibrium.ip = 7.8e6
ini.equilibrium.κ = 1.84
ini.equilibrium.R0 = 3.3
ini.equilibrium.ϵ = 0.34
ini.equilibrium.δ = 0.5 # not defined in paper
ini.equilibrium.ζ = 0.0 # not defined in paper

# Set so ne0, ne_ped, and <ne> all match paper
#   but fGW is a bit high
ini.core_profiles.ne_setting = :greenwald_fraction
ini.core_profiles.ne_value = 0.73
ini.core_profiles.ne_core_to_ped_ratio = 1.73
ini.core_profiles.ne_sep_to_ped_ratio = 0.96
ini.core_profiles.ne_shaping = 1.1

# Set so Te0 and <Te> match paper,
#   Te_ped is higher than 4e3 from paper
ini.core_profiles.Te_core = 27E3
ini.core_profiles.Te_ped = 7E3
ini.core_profiles.Te_shaping = 1.1

ini.core_profiles.w_ped = 0.05
ini.core_profiles.zeff = 1.2
ini.core_profiles.helium_fraction = 0.05

ini.ic_antenna[1].power_launched = 13.6 * 1e6 #rf power coupled

resize!(ini.lh_antenna, 1)
ini.lh_antenna[1].power_launched = 25 * 1e6 #rf power coupled

if true
    ini.build.layers = FUSE.layers_meters_from_fractions(;
            lfs_multiplier=1.0,
            wall=0.1,
            blanket=1.0,
            shield=0.5,
            vessel=0.2,
            pf_inside_tf=false,
            pf_outside_tf=true,
            thin_vessel_walls=true)
    ini.build.layers[:OH].coils_inside = 1
    #ini.build.layers[:gap_lfs_coils].coils_inside = 2  # use if pf_insdie_tf=true
    ini.build.layers[:lfs_shield].coils_inside = 2    # two coils inside TF
    ini.build.layers[:gap_cryostat].coils_inside = 4  # four coils outside TF

else
    # This tries to match the radial build from paper, but HFSsizing won't converge
    plug = 0.1
    cs = 0.7 - plug
    tf = 1.34 - 0.7
    gap = 1.35 - 1.34
    shield = 1.89 - 1.35
    tank = 1.92 - 1.89
    blanket = 2.12 - 1.92
    wall = 2.2 - 2.12
    ini.build.layers = FUSE.OrderedCollections.OrderedDict(
            :gap_plug => plug,
            :OH => cs,
            :hfs_TF => tf,
            :hfs_gap_TF_shield => gap,
            :hfs_shield => shield,
            :hfs_vacuum_vessel => tank,
            :hfs_blanket => blanket,
            :hfs_first_wall => wall,
            :plasma => 2.25,
            :lfs_first_wall => wall,
            :lfs_blanket => 2 * blanket,
            :lfs_vacuum_vessel => tank,
            :lfs_shield => shield,
            :lfs_gap_TF_shield => 10 * gap,
            :lfs_TF => tf,
            :gap_cryostat_TF => 15 * gap,
            :cryostat => tank
        )
    ini.build.layers[:OH].coils_inside = 1
    ini.build.layers[:lfs_gap_TF_shield].coils_inside = 2
    ini.build.layers[:gap_cryostat_TF].coils_inside = 4
end

ini.build.n_first_wall_conformal_layers = 1

ini.build.layers[:hfs_blanket].material = :flibe
ini.build.layers[:lfs_blanket].material = :flibe

act.ActorStationaryPlasma.max_iterations = 5
act.ActorStationaryPlasma.convergence_error = 5e-2

act.ActorEquilibrium.model = :FRESCO
act.ActorFRESCO.tolerance = 1e-6
#act.ActorEquilibrium.symmetrize=true

#act.ActorCoreTransport.model = :FluxMatcher
#act.ActorTGLF.model = :TJLF
#act.ActorFluxMatcher.max_iterations = 200
#act.ActorTGLF.tglfnn_model = "sat3_em_fpp"

act.ActorPedestal.model = :none
act.ActorPedestal.density_match = :ne_line # needed for ini.core_profiles.ne_setting = :greenwald_fraction

act.ActorCosting.construction_start_year = 2014.0 ± 0.0;

In [ ]:
dd = IMAS.dd()
FUSE.init!(dd,ini,act);

# roughly mimic paper sources but not refined
act.ActorSimpleIC.actuator[1].width = 0.25
act.ActorSimpleLH.actuator[1].rho_0 = 0.6
act.ActorSimpleLH.actuator[1].width = 0.15

@checkin :init dd ini act;

In [ ]:
@checkout :init dd ini act
FUSE.ActorWholeFacility(dd, act);

In [ ]:
p1 = plot(dd.costing)
display(p1)
savefig(p1, "ARC_costing.pdf")
p2 = plot(dd.build, size=(500,600))
display(p2)
savefig(p2, "ARC_build.pdf")

In [ ]:
# BCL 3/11/26
#  Volume a bit low
#  Fusion power low
#  TBR *very* low and the wall looks weird - I think ActorBlanket isn't converging 
# 
#  N.B.: Varying things above sometimes make the external PF coils 
#        pull the x-point and get very large

FUSE.digest(dd)